# Import Library

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
CORPUS_NO = 1

# Load Dataset, Label Encoder & Tokenizer

Dataset

In [ ]:
train_df = pd.read_csv(f"preprocessed/{CORPUS_NO}/train_bert.csv")
test_df = pd.read_csv(f"preprocessed/{CORPUS_NO}/test_bert.csv")

In [ ]:
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

Label Encoder

In [ ]:
le = joblib.load("preprocessed/label_encoder.pkl")
class_names = list(le.classes_)
NUM_LABELS = len(class_names)

print(class_names)

Tokenizer

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# Tokenization

In [ ]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

In [ ]:
train_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

# DL Things

## Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

## Compute Metrics

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=1
    )

    return {

        "accuracy":
        accuracy_score(
            labels,
            preds
        ),

        "precision":
        precision_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        ),

        "recall":
        recall_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        ),

        "f1":
        f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        )

    }

## Training Arguments

In [ ]:
training_args = TrainingArguments(

    output_dir="indobert_output",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1"
)

## Trainer

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=test_ds,

    compute_metrics=compute_metrics

)

# Modeling

Training

In [ ]:
trainer.train()

Predict

In [ ]:
predictions = trainer.predict(
    test_ds
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = predictions.label_ids

# Evaluation

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(
    figsize=(6,5)
)

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=class_names,

    yticklabels=class_names

)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Actual"
)

plt.title(
    "Confusion Matrix - IndoBERT"
)

plt.show()

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
acc = accuracy_score(
    y_true,
    y_pred
)

prec = precision_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

rec = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

print("\n=== IndoBERT ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

In [ ]:
result = pd.DataFrame([
    {
        "Train_Size": len(train_df),
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    }
])

result

In [ ]:
os.makedirs(
    "results",
    exist_ok=True
)

RESULT_FILE = "results/indobert_results.csv"

if os.path.exists(
    RESULT_FILE
):

    previous = pd.read_csv(
        RESULT_FILE
    )

    result = pd.concat(
        [previous, result],
        ignore_index=True
    )

    result = result.drop_duplicates(
        subset=["Train_Size"],
        keep="last"
    )

result = result.sort_values(
    by="Train_Size"
)

result.to_csv(
    RESULT_FILE,
    index=False
)

print(result)

In [ ]:
model.save_pretrained(
    f"models/{CORPUS_NO}/indobert_model"
)

tokenizer.save_pretrained(
    f"models/{CORPUS_NO}/indobert_model"
)